In [1]:
import torch
import sys
import transformers
import torch
import circuitsvis as cv
import torch.nn as nn
import numpy as np
import einops
from copy import deepcopy
from fancy_einsum import einsum
import transformer_lens.utils as utils
from transformer_lens import HookedTransformer, FactoredMatrix, HookedTransformerConfig
from jaxtyping import Float, Int
from torch import Tensor
import huggingface_hub
from tqdm import tqdm
import torch.nn.functional as F
from transformer_lens.ActivationCache import ActivationCache
import re
from typing import List, Optional
import argparse
import os
from datetime import datetime
import json

# Add the MIB circuit track to the path
sys.path.append('../../../../')
from MIB_circuit_track.dataset import HFEAPDataset
import dotenv
dotenv.load_dotenv()

TOKEN = os.getenv("TOKEN")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.set_grad_enabled(False)

huggingface_hub.login(token=TOKEN)

In [2]:
model = HookedTransformer.from_pretrained("gpt2-small", device=DEVICE, torch_dtype=torch.float32)

Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
# Load dataset using MIB circuit track
hf_task_name = f'mib-bench/ioi'
dataset = HFEAPDataset(hf_task_name, model.tokenizer, split="train", task="ioi", 
                          num_examples=16, counterfactual_type="s2_io_flip_counterfactual")

In [7]:
dataset[15]

('As London and Nick left the terminal, London gave a piece of chalk to',
 'As London and Nick left the terminal, Nick gave a piece of chalk to',
 [8047, 3576])

In [9]:
dataloader = dataset.to_dataloader(batch_size=16)

In [10]:
# Get a batch from the dataloader
dataloader = dataset.to_dataloader(batch_size=16)
batch = next(iter(dataloader))
clean_prompts, corrupted_prompts, labels = batch

# Tokenize the prompts to get tensors
clean_tokens = model.tokenizer(clean_prompts, padding=True, truncation=True, return_tensors="pt")
corrupted_tokens = model.tokenizer(corrupted_prompts, padding=True, truncation=True, return_tensors="pt")

# Convert labels to tensor
labels_tensor = torch.tensor(labels)  # Shape: [batch_size, 2] for IOI task

# Now you have:
# clean_tokens.input_ids: tensor of shape [batch_size, seq_len]
# corrupted_tokens.input_ids: tensor of shape [batch_size, seq_len] 
# labels_tensor: tensor of shape [batch_size, 2]

In [14]:
for prompt in clean_prompts:
    print(prompt)
    print(model.to_str_tokens(prompt))
    print(len(model.to_str_tokens(prompt)))
    print()
    

As Carl and Maria left the consulate, Carl gave a fridge to
['<|endoftext|>', 'As', ' Carl', ' and', ' Maria', ' left', ' the', ' consulate', ',', ' Carl', ' gave', ' a', ' fridge', ' to']
14

After Kevin and Bob spent some time at the racecourse, Kevin offered a duster to
['<|endoftext|>', 'After', ' Kevin', ' and', ' Bob', ' spent', ' some', ' time', ' at', ' the', ' race', 'course', ',', ' Kevin', ' offered', ' a', ' d', 'uster', ' to']
19

After Brian and Matt spent some time at the vet, Brian offered a button to
['<|endoftext|>', 'After', ' Brian', ' and', ' Matt', ' spent', ' some', ' time', ' at', ' the', ' vet', ',', ' Brian', ' offered', ' a', ' button', ' to']
17

After Brad and Louis went to the room, Brad gave a picture frame to
['<|endoftext|>', 'After', ' Brad', ' and', ' Louis', ' went', ' to', ' the', ' room', ',', ' Brad', ' gave', ' a', ' picture', ' frame', ' to']
16

While Jean and Martin were working at the meeting, Jean gave a plant to
['<|endoftext|>', 'While', '